In [ ]:
# ============================================================
# APPROACH 2 — MarianMT Urdu→English
# opus-mt-ur-en, OPUS-100, correct direction throughout
# ============================================================

# ── CELL 1: Install ──────────────────────────────────────────
!pip install transformers datasets sacrebleu sentencepiece sacremoses -q
%pip install -q transformers datasets sacrebleu[ja] sentencepiece sacremoses

# ── CELL 2: Imports ──────────────────────────────────────────
import os
import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from sacrebleu.metrics import BLEU, CHRF

Note: you may need to restart the kernel to use updated packages.


In [2]:

# ── CELL 3: Config ───────────────────────────────────────────
MODEL_NAME   = "Helsinki-NLP/opus-mt-ur-en"   # Urdu→English, correct direction
MAX_LENGTH   = 128
BATCH_SIZE   = 32
LR           = 2e-5
EPOCHS       = 3
WARMUP_STEPS = 200
OUTPUT_DIR   = "./marianmt-ur-en-finetuned"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [3]:

# ── CELL 4: Load model ───────────────────────────────────────
print("Loading opus-mt-ur-en...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model     = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
print(f"Vocab size  : {tokenizer.vocab_size:,}")
print(f"Params      : {sum(p.numel() for p in model.parameters()):,}")

Loading opus-mt-ur-en...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/848k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/816k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Vocab size  : 62,025
Params      : 139,933,184


In [4]:
# ── CELL 5: Load OPUS-100 data ───────────────────────────────
# Source = Urdu, Target = English throughout
print("\nLoading OPUS-100 ur-en...")
dataset = load_dataset("Helsinki-NLP/opus-100", "en-ur")

# FIX: ur is source, en is target — consistent everywhere
train_ur = [ex["translation"]["ur"] for ex in dataset["train"]]
train_en = [ex["translation"]["en"] for ex in dataset["train"]]
val_ur   = [ex["translation"]["ur"] for ex in dataset["validation"]]
val_en   = [ex["translation"]["en"] for ex in dataset["validation"]]
test_ur  = [ex["translation"]["ur"] for ex in dataset["test"]]
test_en  = [ex["translation"]["en"] for ex in dataset["test"]]

# Cap training data
train_ur = train_ur[:50000]
train_en = train_en[:50000]

print(f"Train: {len(train_ur):,} | Val: {len(val_ur):,} | Test: {len(test_ur):,}")

# Sanity check — should show Urdu then English
print("\nSample check (should be Urdu source, English target):")
for i in range(2):
    print(f"  UR: {train_ur[i]}")
    print(f"  EN: {train_en[i]}")
    print()


Loading OPUS-100 ur-en...


README.md: 0.00B [00:00, ?B/s]

en-ur/test-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

en-ur/train-00000-of-00001.parquet:   0%|          | 0.00/148M [00:00<?, ?B/s]

en-ur/validation-00000-of-00001.parquet:   0%|          | 0.00/296k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/753913 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train: 50,000 | Val: 2,000 | Test: 2,000

Sample check (should be Urdu source, English target):
  UR: اورجب ہم نے موسیٰ سے چالیس رات کا وعدہ کیا پھر اس کے بعد تم نے بچھڑا بنا لیا حالانکہ تم ظالم تھے
  EN: Yet, remember, as We communed with Moses for forty nights you took the calf in his absence (and worshipped it), and you did wrong.

  UR: بجز تیرے اُن بندوں کے جنہیں تو نے خالص کر لیا ہے"
  EN: Except, among them, Your chosen servants."



In [5]:

# ── CELL 6: Translate function ───────────────────────────────
def translate_batch(urdu_sentences, mdl, tok, batch_size=32, max_len=128):
    """Translate Urdu→English in batches."""
    mdl.eval()
    translations = []
    for i in tqdm(range(0, len(urdu_sentences), batch_size), desc="Translating"):
        batch  = urdu_sentences[i : i + batch_size]
        inputs = tok(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len,
        ).to(device)
        with torch.no_grad():
            outputs = mdl.generate(
                **inputs,
                num_beams=4,
                max_length=max_len,
                early_stopping=True,
            )
        translations.extend(tok.batch_decode(outputs, skip_special_tokens=True))
    return translations

In [10]:

# ── CELL 7: Baseline evaluation ──────────────────────────────
print("=" * 60)
print("STEP 1: BASELINE EVALUATION (no fine-tuning)")
print("=" * 60)

bleu_metric = BLEU(tokenize="13a", effective_order=True)
chrf_metric = CHRF(word_order=2)

# Evaluate on test set (Urdu in, English out, compare to English refs)
baseline_hyp  = translate_batch(test_ur, model, tokenizer, BATCH_SIZE)
baseline_bleu = bleu_metric.corpus_score(baseline_hyp, [test_en])
baseline_chrf = chrf_metric.corpus_score(baseline_hyp, [test_en])

print(f"\nBaseline BLEU  : {baseline_bleu.score:.2f}")
print(f"Baseline ChrF++: {baseline_chrf.score:.4f}")
print(f"\n(Basit et al. 2024 reported BLEU=12.06 on Tatoeba)")

STEP 1: BASELINE EVALUATION (no fine-tuning)


Translating: 100%|██████████| 63/63 [02:16<00:00,  2.17s/it]



Baseline BLEU  : 25.35
Baseline ChrF++: 44.8637

(Basit et al. 2024 reported BLEU=12.06 on Tatoeba)


In [12]:
# ── CELL 8: Error analysis on baseline ───────────────────────
print("\n" + "=" * 60)
print("STEP 2: ERROR ANALYSIS (baseline)")
print("=" * 60)

sent_scores = [
    bleu_metric.sentence_score(h, [r]).score
    for h, r in zip(baseline_hyp, test_en)
]
worst_idx = np.argsort(sent_scores)[:10]

error_rows = []
for rank, idx in enumerate(worst_idx):
    print(f"--- Example {rank+1} ---")
    print(f"  Source (UR)  : {test_ur[idx]}")
    print(f"  Reference    : {test_en[idx]}")
    print(f"  Hypothesis   : {baseline_hyp[idx]}")
    print(f"  Sent BLEU    : {sent_scores[idx]:.2f}\n")
    error_rows.append({
        "urdu_source" : test_ur[idx],
        "reference"   : test_en[idx],
        "hypothesis"  : baseline_hyp[idx],
        "sent_bleu"   : sent_scores[idx],
    })

pd.DataFrame(error_rows).to_csv("error_analysis_baseline.csv",
                                index=False, encoding="utf-8-sig")


STEP 2: ERROR ANALYSIS (baseline)
--- Example 1 ---
  Source (UR)  : اس ویڈیو پر مختلف ردعمل دیکھئے گئے۔
  Reference    : Mexico: Children in Viral Video Shake Up Presidential Campaign
  Hypothesis   : Many reactions on this video have been seen on this video.
  Sent BLEU    : 0.00

--- Example 2 ---
  Source (UR)  : paper size
  Reference    : paper size
  Hypothesis   : SUVARIA
  Sent BLEU    : 0.00

--- Example 3 ---
  Source (UR)  : paper size
  Reference    : paper size
  Hypothesis   : SUVARIA
  Sent BLEU    : 0.00

--- Example 4 ---
  Source (UR)  : ٓA3paper size
  Reference    : paper size
  Hypothesis   : Come on the 'Sptics'
  Sent BLEU    : 0.00

--- Example 5 ---
  Source (UR)  : paper size
  Reference    : paper size
  Hypothesis   : SUVARIA
  Sent BLEU    : 0.00

--- Example 6 ---
  Source (UR)  : paper size
  Reference    : paper size
  Hypothesis   : SUVARIA
  Sent BLEU    : 0.00

--- Example 7 ---
  Source (UR)  : paper size
  Reference    : paper size
  Hypothesis   

In [13]:
# ── CELL 9: Tokenize for fine-tuning ─────────────────────────
print("\nTokenizing...")

class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, src_list, tgt_list, tok, max_len=128):
        model_inputs = tok(
            src_list,
            text_target=tgt_list,
            max_length=max_len,
            truncation=True,
            padding="max_length",
        )
        # mask padding in labels
        model_inputs["labels"] = [
            [(t if t != tok.pad_token_id else -100) for t in label]
            for label in model_inputs["labels"]
        ]
        self.enc = model_inputs

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.enc.items()}

train_dataset = TranslationDataset(train_ur, train_en, tokenizer, MAX_LENGTH)
val_dataset   = TranslationDataset(val_ur,   val_en,   tokenizer, MAX_LENGTH)
print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")



Tokenizing...
Train: 50,000 | Val: 2,000


In [14]:


# ── CELL 10: compute_metrics ─────────────────────────────────
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = [l.strip() for l in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    return {
        "sacrebleu": bleu_metric.corpus_score(decoded_preds, [decoded_labels]).score,
        "chrf_pp"  : chrf_metric.corpus_score(decoded_preds, [decoded_labels]).score,
    }

In [15]:

# ── CELL 11: Training args ───────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = LR,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    num_train_epochs            = EPOCHS,
    warmup_steps                = WARMUP_STEPS,
    weight_decay                = 0.01,
    predict_with_generate       = True,
    generation_max_length       = MAX_LENGTH,
    load_best_model_at_end      = True,
    metric_for_best_model       = "sacrebleu",
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = True,
    save_total_limit            = 2,
    report_to                   = "none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)


In [16]:
# ── CELL 12: Train ───────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: FINE-TUNING")
print("=" * 60)
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Training loss: {train_result.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



STEP 3: FINE-TUNING


/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Sacrebleu,Chrf Pp
1,0.522164,0.888708,22.797363,43.829788
2,0.476499,0.901644,21.258141,42.817731
3,0.445250,0.905063,21.115635,42.645107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training loss: 0.4850


In [17]:
# ── CELL 13: Evaluate fine-tuned model ───────────────────────
print("\n" + "=" * 60)
print("STEP 4: FINE-TUNED EVALUATION")
print("=" * 60)

finetuned_model = MarianMTModel.from_pretrained(OUTPUT_DIR).to(device)
finetuned_hyp   = translate_batch(test_ur, finetuned_model, tokenizer, BATCH_SIZE)
finetuned_bleu  = bleu_metric.corpus_score(finetuned_hyp, [test_en])
finetuned_chrf  = chrf_metric.corpus_score(finetuned_hyp, [test_en])

print(f"\nBaseline   BLEU: {baseline_bleu.score:.2f}  ChrF++: {baseline_chrf.score:.4f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}  ChrF++: {finetuned_chrf.score:.4f}")
print(f"Delta BLEU     : {finetuned_bleu.score - baseline_bleu.score:+.2f}")


STEP 4: FINE-TUNED EVALUATION


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Translating: 100%|██████████| 63/63 [02:20<00:00,  2.23s/it]



Baseline   BLEU: 25.35  ChrF++: 44.8637
Fine-tuned BLEU: 21.40  ChrF++: 42.5453
Delta BLEU     : -3.95


In [18]:
# ── CELL 14: Results table ───────────────────────────────────
results = pd.DataFrame([
    {"Model": "Basit et al. 2024 (published)",     "BLEU": 12.06, "ChrF++": 0.39,  "Note": "published baseline"},
    {"Model": "MarianMT ur-en baseline (ours)",     "BLEU": round(baseline_bleu.score, 2),  "ChrF++": round(baseline_chrf.score, 4),  "Note": "our reproduction"},
    {"Model": "MarianMT ur-en fine-tuned (ours)",   "BLEU": round(finetuned_bleu.score, 2), "ChrF++": round(finetuned_chrf.score, 4), "Note": "our system"},
    {"Model": "IndicTrans2 SOTA",                   "BLEU": 30.76, "ChrF++": 0.53,  "Note": "upper bound"},
])
print("\n", results.to_string(index=False))
results.to_csv("results_table.csv", index=False)


                            Model  BLEU  ChrF++               Note
   Basit et al. 2024 (published) 12.06  0.3900 published baseline
  MarianMT ur-en baseline (ours) 25.35 44.8637   our reproduction
MarianMT ur-en fine-tuned (ours) 21.40 42.5453         our system
                IndicTrans2 SOTA 30.76  0.5300        upper bound


In [19]:
# ── CELL 15: Comparative error analysis ──────────────────────
comparison_rows = []
for rank, idx in enumerate(worst_idx):
    comparison_rows.append({
        "urdu_source" : test_ur[idx],
        "reference"   : test_en[idx],
        "baseline"    : baseline_hyp[idx],
        "finetuned"   : finetuned_hyp[idx],
    })
pd.DataFrame(comparison_rows).to_csv("error_analysis_comparison.csv",
                                     index=False, encoding="utf-8-sig")
print("\nAll files saved.")
print(f"Baseline BLEU  : {baseline_bleu.score:.2f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}")


All files saved.
Baseline BLEU  : 25.35
Fine-tuned BLEU: 21.40
